In [1]:
import pyreadstat
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [2]:
# Path to sav file
crisk_bg_sav = f"/home/eprashar_solutions_corelogic_com/crime-idx-2026/data/ns/location_inc_ns4_2025q4_block_group_data.sav"
# Read the sav file
crisk_bg_df, meta = pyreadstat.read_sav(crisk_bg_sav)
crisk_bg_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300363 entries, 0 to 300362
Data columns (total 45 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   lon                      300363 non-null  float64
 1   lat                      300363 non-null  float64
 2   bg_key                   300363 non-null  str    
 3   ct_key                   300363 non-null  str    
 4   country_code             300363 non-null  str    
 5   population               300363 non-null  float64
 6   assault_pt_ct            300363 non-null  float64
 7   assault_pt_fut           300363 non-null  float64
 8   assault_pt_past          300363 non-null  float64
 9   burglary_pt_ct           300363 non-null  float64
 10  burglary_pt_fut          300363 non-null  float64
 11  burglary_pt_past         300363 non-null  float64
 12  larceny_pt_ct            300363 non-null  float64
 13  larceny_pt_fut           300363 non-null  float64
 14  larceny_pt_past

In [3]:
#Examine columns and a few rows
print(f"Columns are {crisk_bg_df.columns.to_list()}")
print("="*80)
print(crisk_bg_df.head(10))

Columns are ['lon', 'lat', 'bg_key', 'ct_key', 'country_code', 'population', 'assault_pt_ct', 'assault_pt_fut', 'assault_pt_past', 'burglary_pt_ct', 'burglary_pt_fut', 'burglary_pt_past', 'larceny_pt_ct', 'larceny_pt_fut', 'larceny_pt_past', 'murder_pt_ct', 'murder_pt_fut', 'murder_pt_past', 'mvt_pt_ct', 'mvt_pt_fut', 'mvt_pt_past', 'rape_pt_ct', 'rape_pt_fut', 'rape_pt_past', 'robbery_pt_ct', 'robbery_pt_fut', 'robbery_pt_past', 'violent_pt_ct', 'vandal_pt_ct', 'property_pt_ct', 'vandal_pt_fut', 'violent_pt_fut', 'property_pt_fut', 'vandal_pt_past', 'violent_pt_past', 'property_pt_past', 'total_pt_ct', 'total_pt_past', 'total_pt_fut', 'fire_pt_ct', 'fire_pt_past', 'fire_pt_fut', 'lite_total_risk_pt_ct', 'lite_total_risk_pt_past', 'lite_total_risk_pt_fut']
         lon        lat        bg_key       ct_key country_code  population  \
0 -86.489661  32.465832  010010201001  01001020100          USA       609.0   
1 -86.489672  32.485873  010010201002  01001020100          USA      1271.0

In [4]:
# Filter our crime risk data to Houston-area block groups
# Harris County FIPS = 48201 (primary), Fort Bend = 48157, Montgomery = 48339
houston_fips_prefixes = ['48201', '48157', '48339']
crisk_houston_df = crisk_bg_df[crisk_bg_df['bg_key'].str[:5].isin(houston_fips_prefixes)].copy()
print(f"Houston-area block groups: {len(crisk_houston_df):,} out of {len(crisk_bg_df):,} total")
print(f"Counties: {crisk_houston_df['bg_key'].str[:5].value_counts().to_dict()}")

Houston-area block groups: 3,524 out of 300,363 total
Counties: {'48201': 2830, '48157': 366, '48339': 328}


In [5]:
# Load Houston crime data from Police Department
houston_county_csv = f"/home/eprashar_solutions_corelogic_com/crime-idx-2026/data/public/houston/NIBRSPublicView2025.csv"
houston_df = pd.read_csv(houston_county_csv)
print("Houston data loaded!")

Houston data loaded!


/tmp/ipykernel_2417728/426193771.py:3: DtypeWarning: Columns (0: ZIP Code) have mixed types. Specify dtype option on import or set low_memory=False.
  houston_df = pd.read_csv(houston_county_csv)


In [6]:
# Examine columns and a few rows
print(f"Columns are {houston_df.columns.to_list()}")
print("="*80)
print(houston_df.info())

Columns are ['Incident', 'Occurrence Date', 'Occurrence Hour', 'NIBRS Class', 'NIBRS Description', 'Offense Count', 'Beat', 'Premise', 'Street Number', 'Street Name', 'Street Type', 'Street Suffix', 'City', 'ZIP Code', 'Map Longitude', 'Map Latitude']
<class 'pandas.DataFrame'>
RangeIndex: 240696 entries, 0 to 240695
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Incident           240696 non-null  int64  
 1   Occurrence Date    240696 non-null  str    
 2   Occurrence Hour    240696 non-null  int64  
 3   NIBRS Class        240696 non-null  str    
 4   NIBRS Description  240696 non-null  str    
 5   Offense Count      240696 non-null  int64  
 6   Beat               240520 non-null  str    
 7   Premise            240696 non-null  str    
 8   Street Number      239754 non-null  str    
 9   Street Name        240696 non-null  str    
 10  Street Type        222742 non-null  str    
 11  Street

In [7]:
# Load TX block group boundaries
tx_bg_gdf = gpd.read_file("/home/eprashar_solutions_corelogic_com/crime-idx-2026/data/boundaries/cb_2025_48_bg_500k.zip")
print(f"TX block groups loaded: {len(tx_bg_gdf):,}")
print(f"Columns: {tx_bg_gdf.columns.tolist()}")
print(f"CRS: {tx_bg_gdf.crs}")

TX block groups loaded: 18,626
Columns: ['STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE', 'GEOIDFQ', 'GEOID', 'NAME', 'NAMELSAD', 'LSAD', 'ALAND', 'AWATER', 'geometry']
CRS: EPSG:4269


In [8]:
# Filter to Houston-area counties only (faster join)
tx_bg_gdf['county_fips'] = tx_bg_gdf['STATEFP'] + tx_bg_gdf['COUNTYFP']
houston_bg_gdf = tx_bg_gdf[tx_bg_gdf['county_fips'].isin(['48201', '48157', '48339'])].copy()
print(f"Houston-area block group polygons: {len(houston_bg_gdf):,}")

Houston-area block group polygons: 3,524


In [9]:
# Drop rows with missing lat/lon
houston_valid = houston_df.dropna(subset=['Map Latitude', 'Map Longitude']).copy()
print(f"Houston records with valid coords: {len(houston_valid):,} / {len(houston_df):,}")

# Create point geometries
geometry = [Point(xy) for xy in zip(houston_valid['Map Longitude'], houston_valid['Map Latitude'])]
houston_gdf = gpd.GeoDataFrame(houston_valid, geometry=geometry, crs="EPSG:4326")

# Ensure same CRS
houston_bg_gdf = houston_bg_gdf.to_crs("EPSG:4326")

# Spatial join
houston_joined = gpd.sjoin(houston_gdf, houston_bg_gdf[['GEOID', 'geometry']], how='left', predicate='within')
houston_joined = houston_joined.rename(columns={'GEOID': 'bg_key'})

# Drop geometry and index_right (from spatial join)
houston_joined = houston_joined.drop(columns=['geometry', 'index_right'])

print(f"Records matched to a block group: {houston_joined['bg_key'].notna().sum():,}")
print(f"Records NOT matched: {houston_joined['bg_key'].isna().sum():,}")
print("Sample of joined data:")
print(houston_joined.head())

Houston records with valid coords: 237,011 / 240,696
Records matched to a block group: 236,764
Records NOT matched: 247
Sample of joined data:
   Incident Occurrence Date  Occurrence Hour NIBRS Class  \
0  37542125        1/1/2025                0         13C   
1  37542125        1/1/2025                0         90J   
2  84089625        1/1/2025                0         26F   
3   6608325        1/1/2025                0         13C   
4   6608325        1/1/2025                0         23H   

           NIBRS Description  Offense Count        Beat  \
0               Intimidation              1   Beat 7C30   
1  Trespass of real property              1   Beat 7C30   
2             Identify theft              1  Beat 20G10   
3               Intimidation              2  Beat 18F60   
4          All other larceny              1  Beat 18F60   

                                Premise Street Number Street Name Street Type  \
0                     Hotel, Motel, ETC          3800     TI

In [10]:
# See what offense categories exist in the data
print(f"Unique NIBRS Class values: {houston_joined['NIBRS Class'].nunique()}")
print("="*80)
print(houston_joined[['NIBRS Class','NIBRS Description']].value_counts())

Unique NIBRS Class values: 60
NIBRS Class  NIBRS Description                        
13B          Simple assault                               24248
23F          Theft from motor vehicle                     23861
290          Destruction, damage, vandalism               20426
13C          Intimidation                                 18577
23H          All other larceny                            17451
23C          Shoplifting                                  16949
90Z          All other offenses                           16643
13A          Aggravated Assault                           13041
240          Motor vehicle theft                          12355
220          Burglary, Breaking and Entering              11284
90J          Trespass of real property                    10964
23G          Theft of motor vehicle parts or accessory     8044
35A          Drug, narcotic violations                     7995
120          Robbery                                       4973
90D          Drivin

In [11]:
# Map NIBRS Class codes to our crime risk model categories
nibrs_to_category = {
    # Assault
    '13A': 'assault',
    # NOTE: 13B and 13C are not included in model mapping
    # '13B': 'assault', '13C': 'assault',
    # Burglary
    '220': 'burglary',
    # Larceny (all theft subtypes)
    '23A': 'larceny', '23B': 'larceny', '23C': 'larceny', '23D': 'larceny',
    '23E': 'larceny', '23F': 'larceny', '23G': 'larceny', '23H': 'larceny',
    # Murder
    '09A': 'murder', '09B': 'murder', '09C': 'murder',
    # Motor Vehicle Theft
    '240': 'mvt',
    # Rape / Sexual assault
    '11A': 'rape', '11B': 'rape', '11C': 'rape', '11D': 'rape',
    '36B': 'rape',
    # Robbery
    '120': 'robbery',
    # Vandalism
    '290': 'vandal',
    # Arson → fire
    '200': 'fire',
}

# Add category column (keeps original NIBRS Class & Description intact)
houston_joined['crime_category'] = houston_joined['NIBRS Class'].map(nibrs_to_category)

# Coverage report
mapped = houston_joined['crime_category'].notna().sum()
total = len(houston_joined)
print(f"Mapped: {mapped:,} / {total:,} ({mapped/total*100:.1f}%)")
print("="*80)
print("Category counts:")
print(houston_joined['crime_category'].value_counts().to_string())
print("="*80)
unmapped = houston_joined.loc[houston_joined['crime_category'].isna(), ['NIBRS Class','NIBRS Description']].value_counts()
print(f"\nUnmapped ({unmapped.sum():,} records across {len(unmapped)} NIBRS classes):")
print(unmapped)

Mapped: 132,286 / 237,011 (55.8%)
Category counts:
crime_category
larceny     67789
vandal      20426
assault     13041
mvt         12355
burglary    11284
robbery      4973
rape         1921
murder        289
fire          208

Unmapped (104,725 records across 38 NIBRS classes):
NIBRS Class  NIBRS Description                      
13B          Simple assault                             24248
13C          Intimidation                               18577
90Z          All other offenses                         16643
90J          Trespass of real property                  10964
35A          Drug, narcotic violations                   7995
90D          Driving under the influence                 4971
520          Weapon law violations                       2910
26F          Identify theft                              2819
26A          False pretenses, swindle                    2341
250          Counterfeiting, forgery                     2316
90C          Disorderly conduct               

In [12]:
# Check if model's property = burglary + larceny + mvt + fire, or just burglary + larceny + mvt
check = crisk_houston_df[['property_pt_ct', 'burglary_pt_ct', 'larceny_pt_ct', 'mvt_pt_ct', 'fire_pt_ct']].head(20)
check['sum_with_fire'] = check['burglary_pt_ct'] + check['larceny_pt_ct'] + check['mvt_pt_ct'] + check['fire_pt_ct']
check['sum_without_fire'] = check['burglary_pt_ct'] + check['larceny_pt_ct'] + check['mvt_pt_ct']
print(check)

        property_pt_ct  burglary_pt_ct  larceny_pt_ct  mvt_pt_ct  fire_pt_ct  \
256599       57.768533        7.566008      36.443404   9.525074       112.0   
256600       54.043834        7.078180      34.093670   8.910933       148.0   
256601       60.593594        7.936009      38.225600   9.990880       186.0   
256602       60.166381        7.880057      37.956092   9.920439       109.0   
256603       45.736261        6.840554      19.659537   8.451208        61.0   
256604       41.044240        6.138791      17.642692   7.584210       145.0   
256605       44.813683        6.194026      19.682235   8.769654        78.0   
256606       24.994469        3.824179      11.632512   4.340737       128.0   
256607       28.822788        4.409915      13.414225   5.005593       136.0   
256608       37.614822        5.755105      17.506067   6.532487        97.0   
256609       30.589539        4.680230      14.236476   5.312421       107.0   
256610       52.995563        7.340400  

In [13]:
# Aggregate: count incidents per block group per crime category
actuals_bg = (houston_joined
    .dropna(subset=['bg_key', 'crime_category'])
    .groupby(['bg_key', 'crime_category'])['Incident']
    .count()
    .reset_index(name='actual_count')
)

# Pivot to wide: one column per category
actuals_wide = actuals_bg.pivot(index='bg_key', columns='crime_category', values='actual_count').fillna(0)
actuals_wide.columns = [f'{c}_actual' for c in actuals_wide.columns]

# Add composite categories to match model
actuals_wide['violent_actual'] = (actuals_wide['assault_actual'] + actuals_wide['murder_actual']
                                  + actuals_wide['rape_actual'] + actuals_wide['robbery_actual'])
actuals_wide['property_actual'] = (actuals_wide['burglary_actual'] + actuals_wide['larceny_actual']
                                   + actuals_wide['mvt_actual'])
actuals_wide['total_actual'] = actuals_wide[[c for c in actuals_wide.columns
                                             if c.endswith('_actual') and c not in
                                             ['violent_actual', 'property_actual', 'total_actual']]].sum(axis=1)

actuals_wide = actuals_wide.reset_index()
print(f"Block groups with actuals: {len(actuals_wide):,}")
print(actuals_wide.head())

Block groups with actuals: 1,872
         bg_key  assault_actual  burglary_actual  fire_actual  larceny_actual  \
0  481576701011             5.0              2.0          0.0            13.0   
1  481576701012             7.0              8.0          1.0            18.0   
2  481576701013             4.0              2.0          0.0             9.0   
3  481576701014            14.0              5.0          0.0            27.0   
4  481576701021             1.0              5.0          0.0             4.0   

   murder_actual  mvt_actual  rape_actual  robbery_actual  vandal_actual  \
0            0.0         1.0          1.0             1.0            7.0   
1            0.0         7.0          1.0             4.0           17.0   
2            0.0         9.0          1.0             2.0            2.0   
3            1.0        16.0          0.0             1.0            8.0   
4            0.0         0.0          0.0             0.0           10.0   

   violent_actual  prop

In [14]:
# Join actuals with model predictions on bg_key
comparison_df = crisk_houston_df.merge(actuals_wide, on='bg_key', how='left')

# Fill NaN actuals with 0 (block groups with no mapped incidents)
actual_cols = [c for c in comparison_df.columns if c.endswith('_actual')]

# NOTE: Instead of filling all NaN actuals with 0, we will focus on block groups where HPD actually operates (i.e., has at least 1 mapped incident). This is because many block groups may have zero incidents simply because they are outside of HPD's jurisdiction, and including them could skew the comparison.
# comparison_df[actual_cols] = comparison_df[actual_cols].fillna(0)

# Only compare block groups where HPD actually operates
comparison_df_hpd = comparison_df[comparison_df[actual_cols].sum(axis=1) > 0].copy()

# Also drop block groups with zero population
comparison_df_hpd = comparison_df_hpd[comparison_df_hpd['population'] > 0].copy()
print(f"Block groups in model: {len(crisk_houston_df):,}")
print(f"Block groups in HPD jurisdiction (approx): {len(comparison_df_hpd):,}")

# Side-by-side summary: model prediction (_pt_ct) vs actual count
categories = ['assault', 'burglary', 'larceny', 'murder', 'mvt', 'rape',
              'robbery', 'vandal', 'fire', 'violent', 'property', 'total']

summary_rows = []
for cat in categories:
    pred_col = f'{cat}_pt_ct'
    actual_col = f'{cat}_actual'
    if pred_col in comparison_df_hpd.columns and actual_col in comparison_df_hpd.columns:
        summary_rows.append({
            'category': cat,
            'model_sum': comparison_df_hpd[pred_col].sum(),
            'actual_sum': comparison_df_hpd[actual_col].sum(),
            'model_mean': comparison_df_hpd[pred_col].mean(),
            'actual_mean': comparison_df_hpd[actual_col].mean(),
            'correlation': comparison_df_hpd[[pred_col, actual_col]].corr(method='spearman').iloc[0, 1]
        })

summary = pd.DataFrame(summary_rows)
summary['ratio'] = summary['actual_sum'] / summary['model_sum']
print("\n" + "="*90)
print(summary.to_string(index=False, float_format='%.3f'))

Block groups in model: 3,524
Block groups in HPD jurisdiction (approx): 1,870

category  model_sum  actual_sum  model_mean  actual_mean  correlation  ratio
 assault  14674.251   13027.000       7.847        6.966        0.558  0.888
burglary  12111.023   11282.000       6.476        6.033        0.435  0.932
 larceny  57573.627   67767.000      30.788       36.239        0.399  1.177
  murder    228.249     288.000       0.122        0.154        0.213  1.262
     mvt  13940.249   12345.000       7.455        6.602        0.309  0.886
    rape   1326.562    1918.000       0.709        1.026        0.210  1.446
 robbery   4525.412    4969.000       2.420        2.657        0.428  1.098
  vandal  39376.247   20418.000      21.057       10.919        0.472  0.519
    fire 335368.564     208.000     179.341        0.111        0.117  0.001
 violent  19122.124   20202.000      10.226       10.803        0.571  1.056
property  89157.333   91394.000      47.678       48.874        0.371  1.0

In [15]:
# Normalize actuals to rate per 1,000 population (to match model units)
# First, bring in population from the model data
actuals_norm = actuals_wide.merge(crisk_houston_df[['bg_key', 'population']], on='bg_key', how='left')

# Convert counts to rates per 1,000
rate_cols = [c for c in actuals_norm.columns if c.endswith('_actual')]
for col in rate_cols:
    rate_col_name = col.replace('_actual', '_rate_actual')
    actuals_norm[rate_col_name] = (actuals_norm[col] / actuals_norm['population']) * 1000

# Drop block groups with zero population (avoid inf)
actuals_norm = actuals_norm[actuals_norm['population'] > 0]

# Keep only bg_key and rate columns
rate_actual_cols = [c for c in actuals_norm.columns if c.endswith('_rate_actual')]
actuals_norm = actuals_norm[['bg_key'] + rate_actual_cols]

print(f"Block groups with normalized actuals: {len(actuals_norm):,}")
print(actuals_norm.head())

Block groups with normalized actuals: 1,870
         bg_key  assault_rate_actual  burglary_rate_actual  fire_rate_actual  \
0  481576701011             4.541326              1.816530          0.000000   
1  481576701012             4.263094              4.872107          0.609013   
2  481576701013             3.401361              1.700680          0.000000   
3  481576701014             4.815961              1.719986          0.000000   
4  481576701021             0.849618              4.248088          0.000000   

   larceny_rate_actual  murder_rate_actual  mvt_rate_actual  rape_rate_actual  \
0            11.807448            0.000000         0.908265          0.908265   
1            10.962241            0.000000         4.263094          0.609013   
2             7.653061            0.000000         7.653061          0.850340   
3             9.287926            0.343997         5.503956          0.000000   
4             3.398471            0.000000         0.000000          0

In [16]:
# Join normalized actuals with model predictions
comparison_norm_df = crisk_houston_df.merge(actuals_norm, on='bg_key', how='left')
rate_actual_cols = [c for c in comparison_norm_df.columns if c.endswith('_rate_actual')]
comparison_norm_hpd = comparison_norm_df[comparison_norm_df[rate_actual_cols].sum(axis=1) > 0].copy()

print(f"Block groups in normalized comparison: {len(comparison_norm_hpd):,}")

# Side-by-side: model rate vs actual rate (both per 1,000 pop)
categories = ['assault', 'burglary', 'larceny', 'murder', 'mvt', 'rape',
              'robbery', 'vandal', 'fire', 'violent', 'property', 'total']

summary_norm_rows = []
for cat in categories:
    pred_col = f'{cat}_pt_ct'
    actual_col = f'{cat}_rate_actual'
    if pred_col in comparison_norm_hpd.columns and actual_col in comparison_norm_hpd.columns:
        summary_norm_rows.append({
            'category': cat,
            'model_sum': comparison_norm_hpd[pred_col].sum(),
            'actual_rate_sum': comparison_norm_hpd[actual_col].sum(),
            'model_mean': comparison_norm_hpd[pred_col].mean(),
            'actual_rate_mean': comparison_norm_hpd[actual_col].mean(),
            'corr_spearman': comparison_norm_hpd[[pred_col, actual_col]].corr(method='spearman').iloc[0, 1],
        })

summary_norm = pd.DataFrame(summary_norm_rows)
summary_norm['ratio'] = summary_norm['actual_rate_sum'] / summary_norm['model_sum']
print("\n" + "="*90)
print("Comparison using rates per 1,000 population:")
print(summary_norm.to_string(index=False, float_format='%.3f'))

Block groups in normalized comparison: 1,870

Comparison using rates per 1,000 population:
category  model_sum  actual_rate_sum  model_mean  actual_rate_mean  corr_spearman  ratio
 assault  14674.251        14070.179       7.847             7.524          0.590  0.959
burglary  12111.023        10056.453       6.476             5.378          0.484  0.830
 larceny  57573.627       156985.410      30.788            83.949          0.455  2.727
  murder    228.249          204.997       0.122             0.110          0.215  0.898
     mvt  13940.249        29049.091       7.455            15.534          0.343  2.084
    rape   1326.562         1641.474       0.709             0.878          0.227  1.237
 robbery   4525.412         4655.125       2.420             2.489          0.454  1.029
  vandal  39376.247        29803.339      21.057            15.938          0.533  0.757
    fire 335368.564          209.031     179.341             0.112          0.119  0.001
 violent  19122.124